# 🎓 DECAID Academy — Webinar Simulator (Optional)

> **Hinweis:** Für die normale Nutzung reicht die [Landingpage](https://floriandecaid.github.io/decaid-academy-masterclass-smart-crm-demo-florian-202605/) — sie triggert den Webhook direkt aus dem Browser. Dieses Notebook ist die **Python-Alternative**, falls du die Pipeline aus einem Backend, Skript oder Notebook-Kontext anstoßen willst.

**Was macht dieses Notebook?**

Es simuliert das Verhalten von zwei Webinar-Teilnehmern (Gabi & Jürgen) und feuert ihre Daten als Webhook-Payload an deinen Langdock-Workflow. So kannst du End-to-End testen: **Sign-up → Web Research → Briefing-Doc in Google Drive**.

## Wichtig: Was die Webinar-Plattform überhaupt weiß

Echte Webinar-Tools (Demio, Livestorm, Zoom) liefern dir **nur diese Felder**:
- Vorname, Nachname, E-Mail (aus dem Anmelde-Formular)
- Firma, Position (aus dem Anmelde-Formular)
- Watch-Time (aus der Streaming-Plattform)

**Was sie NICHT wissen:** Jahresumsatz, Branche, aktuelle News, strategische Themen. Das ergänzt die KI im Langdock-Workflow per Web-Search.

## Was du brauchst (5 Minuten Setup)

1. Eine **Langdock Webhook URL** — bekommst du, indem du einen neuen Workflow anlegst und den Webhook-Trigger auf "Copy URL" klickst.
2. Google Drive in Langdock verbunden (für die Doc-Outputs).
3. *(Optional)* Einen **HubSpot Private Access Token** (`pat-...`) — nur falls du die Daten zusätzlich live in HubSpot syncen willst.

## Was dieses Notebook **nicht** macht

- Es schreibt keine echten Kundendaten ins CRM.
- Es nutzt nur fiktive Personas.
- Du kannst es bedenkenlos in einer Live-Demo ausführen.

---

## 1️⃣ Konfiguration

Trage hier deine Werte ein. **Niemals committen** — bei Colab landet das ohnehin nur in deinem Drive.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔧 BITTE HIER DEINE WERTE EINTRAGEN
# ─────────────────────────────────────────────────────────────

# Pflicht: Langdock Webhook (Copy aus dem Webhook-Node deines Workflows)
LANGDOCK_WEBHOOK_URL = "https://api.langdock.com/webhook/<YOUR_WEBHOOK_ID>"

# Optional: HubSpot Private Access Token (beginnt mit 'pat-...')
# Leer lassen, wenn du nur Langdock testest.
HUBSPOT_PAT = ""

# ─────────────────────────────────────────────────────────────
import json, urllib.request, urllib.error

if LANGDOCK_WEBHOOK_URL.startswith("https://api.langdock.com/webhook/<"):
    print("⚠️  Du hast die Webhook-URL noch nicht eingetragen.")
    print("   Öffne Langdock → dein Workflow → Webhook-Node → 'Copy URL'.")
else:
    print("✓ Webhook-URL gesetzt.")

if HUBSPOT_PAT and not HUBSPOT_PAT.startswith("pat-"):
    print("⚠️  HubSpot PAT sollte mit 'pat-' beginnen.")

## 2️⃣ Mock-Personas

Zwei fiktive Webinar-Teilnehmer:

| Persona | Firma | Watch Time | Erwarteter Outcome |
|---|---|---|---|
| **Jürgen** | Kleinhandel Müller GmbH | 3 Min | No-Show → Re-Engagement-Doc im Marketing-Ordner |
| **Gabi** | Gerolsteiner Brunnen GmbH | 58 Min | Hot → Sales-Briefing-Doc im Sales-Ordner |

**Beide Leads werden recherchiert.** Der Workflow holt Revenue + News selbst aus dem Web — egal ob jemand schaut oder nicht. Die Watch-Time entscheidet danach, **in welchem Drive-Ordner** das Briefing-Doc landet.

**Wichtig:** Beide Personas sind erfunden. Gerolsteiner als Firma ist real (deshalb googlebar für die Web-Search-Demo), aber "Gabi Schneider" arbeitet dort nicht.


In [ ]:
# Achtung: Webinar-Tools liefern KEINEN Jahresumsatz, KEINE Branche, KEINE News.
# Nur: Vorname, Nachname, Email, Firma, Position, Watch-Time.
# Alles andere ergänzt der Web-Search-Agent im Langdock-Workflow.

PERSONAS = {
    "juergen": {
        "event": "webinar_engagement_update",
        "lead": {
            "firstname": "Jürgen",
            "lastname": "Müller",
            "email": "juergen.mueller@kleinhandel-mueller.example",
            "company_name": "Kleinhandel Müller GmbH",
            "jobtitle": "Geschäftsführer",
            "webinar_watch_time_minutes": 3,
        },
    },
    "gabi": {
        "event": "webinar_engagement_update",
        "lead": {
            "firstname": "Gabi",
            "lastname": "Schneider",
            "email": "g.schneider@gerolsteiner.example",
            "company_name": "Gerolsteiner Brunnen GmbH & Co. KG",
            "jobtitle": "VP of RevOps",
            "webinar_watch_time_minutes": 58,
        },
    },
}

print(json.dumps(PERSONAS, indent=2, ensure_ascii=False))

## 3️⃣ Webhook-Sender

Schickt eine Persona-Payload per HTTP POST an Langdock.

In [ ]:
def fire(persona_key: str) -> None:
    """Sendet die gewählte Persona an den Langdock-Webhook."""
    payload = PERSONAS.get(persona_key)
    if not payload:
        print(f"❌ Persona '{persona_key}' nicht gefunden.")
        return

    lead = payload['lead']
    print(f"📤 Sende {lead['firstname']} ({lead['company_name']}, {lead['webinar_watch_time_minutes']} Min Watch Time)...")

    body = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(
        LANGDOCK_WEBHOOK_URL,
        data=body,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as res:
            print(f"✓ HTTP {res.status} — Webhook angenommen.")
            print("   → Langdock-Workflow läuft. KI recherchiert jetzt die Firma.")
            print("   → Briefing-Doc erscheint in 20–60 Sekunden im passenden Drive-Ordner.")
    except urllib.error.HTTPError as e:
        print(f"❌ HTTP {e.code}: {e.reason}")
        print(f"   Body: {e.read().decode('utf-8', errors='replace')[:500]}")
    except Exception as e:
        print(f"❌ Fehler: {e}")

## 4️⃣ ❄️ Trigger Jürgen (No-Show, Watch Time = 3 Min)

Erwartung: Workflow recherchiert Kleinhandel Müller GmbH → erkennt: kleiner Lead, hat nicht geschaut → erstellt **Re-Engagement-Doc** im Marketing-Ordner mit personalisierter Wiederansprache, die auf Themen seiner Firma aufbaut.

In [ ]:
fire("juergen")

## 5️⃣ 🔥 Trigger Gabi (Full Watch, 58 Min)

Erwartung: Workflow recherchiert Gerolsteiner Brunnen → findet aktuelle News + Revenue → erkennt: Top-ICP, hat fast komplett geschaut → erstellt **Sales-Briefing-Doc** im Sales-Ordner mit Eisbrecher für den Call.

In [ ]:
fire("gabi")

## 6️⃣ Bonus — HubSpot Sync (optional, kommentiert)

Wenn du den Lead **zusätzlich** in HubSpot anlegen willst, entkommentiere die nächste Zelle. Sonst überspring sie einfach.

> ⚠️ In der Live-Demo lassen wir das aus — wir wollen keine Mock-Personas in einem produktiven HubSpot-Account haben.

In [ ]:
# def sync_to_hubspot(persona_key: str) -> None:
#     """Legt den Lead in HubSpot als Contact an."""
#     if not HUBSPOT_PAT.startswith("pat-"):
#         print("❌ HUBSPOT_PAT nicht gesetzt — Sync übersprungen.")
#         return
#
#     lead = PERSONAS[persona_key]["lead"]
#     body = json.dumps({
#         "properties": {
#             "firstname": lead["firstname"],
#             "lastname": lead["lastname"],
#             "email": lead["email"],
#             "company": lead["company_name"],
#             "jobtitle": lead["jobtitle"],
#             "webinar_watch_time_minutes": lead["webinar_watch_time_minutes"],
#         }
#     }).encode("utf-8")
#
#     req = urllib.request.Request(
#         "https://api.hubapi.com/crm/v3/objects/contacts",
#         data=body,
#         headers={
#             "Content-Type": "application/json",
#             "Authorization": f"Bearer {HUBSPOT_PAT}",
#         },
#         method="POST",
#     )
#     with urllib.request.urlopen(req, timeout=30) as res:
#         print(f"✓ HubSpot Contact angelegt: HTTP {res.status}")
#
# # sync_to_hubspot("gabi")

## ✅ Fertig

- Du hast zwei Mock-Leads an deinen Langdock-Workflow geschickt.
- Schau in die zwei Drive-Ordner, die du im Workflow als Empfänger eingetragen hast.
- **Bei Jürgen:** Re-Engagement-Doc im Marketing-Ordner — personalisierte Wiederansprache, baut auf aktuellen Themen seiner Firma auf.
- **Bei Gabi:** Sales-Briefing-Doc im Sales-Ordner — komplette Recherche zu Gerolsteiner inklusive Eisbrecher für den Call.

**Beide Docs erscheinen.** Das ist der Punkt: kein Lead fällt durchs Raster, jeder bekommt die richtige Behandlung.

**Probleme?** → siehe `materials/quick-start.html` im Repo.